# Ejercicios - Clase 3 -

## Modules

In [6]:
import numpy as np
import ctypes

## Ejercicios

### Ej. 1:

Ejecutar el siguiente programa para explorar el paquete `sys`. Contrastar los resultados con sobre distintas plataformas (si es possible). 

In [ ]:
n = 50
import sys
data = []
for k in range(n):
    a = len(data)
    b = sys.getsizeof(data)
    print("Length: {0:3d}; Size in bytes: {1:4d}".format(a, b))
    data.append(None)

### Ej. 2:

**Triangulo:** 

Definir una clase `triangulo` cómo una tupla de 3 puntos. La clase debe contener los siguientes métodos:

- Calcular el perimetro del triangulo. 
- Clacular el area del triangulo.
- Chequear si el triagulo es rectangulo. 

Deben recordar validar que los 3 puntos conformen un triangulo. 

#### Solución

In [ ]:
import math

class triangle():
    
    def __init__(self,content):
        if type(content)!=type((0,1,2)):
            raise TypeError("El argumento debe ser una tupla de puntos")
        elif len(content)!=3:
            raise ValueError("Debe haber al menos 3 puntos")
        else:  
            self.content=content
    
    def perimeter(self):
        A,B,C=self.content #Define A,B,C to be the points defining your triangle
        u=(B[0]-A[0],B[1]-A[1]) #u=the vector AB
        v=(C[0]-A[0],C[1]-A[1]) #v=the vector AC
        w=(C[0]-B[0],C[1]-B[1]) #w=the vector BC
        return normvect(u)+normvect(v)+normvect(w)
    
    
    #To compute the area of a triangle see the picture above 
    
    def area(self):
        A,B,C=self.content #Define A,B,C to be the points defining your triangle
        u=(B[0]-A[0],B[1]-A[1])  #u=the vector AB
        v=(C[0]-A[0],C[1]-A[1])  #v=the vector AC
        scalarprod_uv=u[0]*v[0]+u[1]*v[1]
        h=math.sqrt(normvect(u)**2-scalarprod_uv**2/normvect(v)**2)
        return normvect(v)*h/2
    
    
    def is_trianglerectangle(self):
        A,B,C=self.content  #Define A,B,C to be the points defining your triangle
        u=(B[0]-A[0],B[1]-A[1]) #u=the vector AB
        v=(C[0]-A[0],C[1]-A[1]) #v=the vector AC
        w=(C[0]-B[0],C[1]-B[1]) #w=the vector BC
        if u>=v and u>=w:
            return v[0]**2+v[1]**2+w[0]**2+w[1]**2==u[0]**2+u[1]**2
        if v>=u and v>=w:
            return u[0]**2+u[1]**2+w[0]**2+w[1]**2==v[0]**2+v[1]**2
        else:
            return v[0]**2+v[1]**2+u[0]**2+u[1]**2==w[0]**2+w[1]**2
        

In [ ]:
Tri=triangle(((0,0),(120,0),(40,200))) #Definition of the triangle
Tri.is_trianglerectangle() #The triangle is not rectangle

### Ej. 3: 

Escribir un programa que simule un ecosistema conteniendo dos tipos de criaturas, **osos** y **peces**. El ecosistema cosiste en un río, el cuál esta modelado como una lista. Cada elemento de la lista debe ser un objeto del tipo `Oso` , del tipo `Pez`, o `None`. Considerando un "contador externo" (random), en cada paso, cada animal intenta moverse hacia una celda adjacente en la lista, o quedarse en su sitio. Si dos animales del mismo tipo estan por ocupar el mismo espacio, deberan quedarse en sus posiciones, y generaremos una nueva criatura de ese tipo en un lugar aleatorio vacio (i.e. `None`) de la lista (se reproducen). Si un oso y un pez ocupan el mismo espacio, entonces el pez desaparecera (es comido).  

#### Solución

Let's define Creature and its subclass first.

In [64]:
from abc import ABCMeta, abstractmethod

# Luego veremos como generar una Metaclase #
# -------------------------------------- #
class Creature(metaclass=ABCMeta):
    
    def __init__(self, ind):
        self.ind = ind  # Index of a creature in ecosystem.
    
    @abstractmethod
    def move(self):
        """Return index to move"""
# ------------------------------------- #
        
        
        
class Bear(Creature):

    def __repr__(self):
        return "Bear(%s)" % self.ind
    
    def move(self):
        to = np.random.choice([-1, 1])
        new_ind = self.ind + to
        print(self, "moves", "left" if to == -1 else "right")
        return new_ind
        
class Fish(Creature):
    
    def __repr__(self):
        return "Fish(%s)" % self.ind
    
    def move(self):
        to = np.random.choice([-1, 1])
        new_ind = self.ind + to
        print(self, "moves", "left" if to == -1 else "right")
        return new_ind       

Definimos la clase de nuestro ecosistema, `River`.

In [65]:
class River:
    
    def __init__(self, n_room):
        self.n_room = n_room
        self.eco = None
    
    def initialize(self):
        self.eco = []
        creatures= np.random.choice([Bear, Fish, None], size=self.n_room)
        for ind, creature in enumerate(creatures):
            self.eco.append(creature(ind) if creature is not None else None)
    
    def next_time_step(self, n=1, verbose=True):
        for i in range(n):
            moving_ind = np.random.choice(list(range(self.n_room)))
            if self.eco[moving_ind] is None:
                print("Nothing happens...")
                pass
            else:
                new_ind = self.eco[moving_ind].move()
                if new_ind < 0 or new_ind > len(self.eco) -1:
                    pass
                elif isinstance(self.eco[moving_ind], Bear):
                    if isinstance(self.eco[new_ind], Bear):
                        pass
                    elif isinstance(self.eco[new_ind], Fish):
                        self.eco[new_ind] = Bear(new_ind)
                        self.eco[moving_ind] = None
                    else:
                        self.eco[new_ind] = Bear(new_ind)
                elif isinstance(self.eco[moving_ind], Fish):
                    if isinstance(self.eco[new_ind], Fish):
                        pass
                    elif isinstance(self.eco[new_ind], Bear):
                        self.eco[moving_ind] = None
                    else:
                        self.eco[new_ind] = Fish(new_ind)
                else:
                    raise ValueError("Undefined Creature")
            if verbose:
                self.display()
        
    def display(self):
        print("===================")
        print("Ecosystem status:\n")
        print(self.eco, "\n")
        print("===================")


In [66]:
river = River(5)

In [67]:
river.initialize()
river.display()

Ecosystem status:

[Bear(0), None, None, Bear(3), Bear(4)] 



In [68]:
river.next_time_step(10)


Bear(0) moves left
Ecosystem status:

[Bear(0), None, None, Bear(3), Bear(4)] 

Nothing happens...
Ecosystem status:

[Bear(0), None, None, Bear(3), Bear(4)] 

Bear(0) moves left
Ecosystem status:

[Bear(0), None, None, Bear(3), Bear(4)] 

Nothing happens...
Ecosystem status:

[Bear(0), None, None, Bear(3), Bear(4)] 

Bear(3) moves left
Ecosystem status:

[Bear(0), None, Bear(2), Bear(3), Bear(4)] 

Bear(0) moves right
Ecosystem status:

[Bear(0), Bear(1), Bear(2), Bear(3), Bear(4)] 

Bear(1) moves left
Ecosystem status:

[Bear(0), Bear(1), Bear(2), Bear(3), Bear(4)] 

Bear(3) moves right
Ecosystem status:

[Bear(0), Bear(1), Bear(2), Bear(3), Bear(4)] 

Bear(4) moves left
Ecosystem status:

[Bear(0), Bear(1), Bear(2), Bear(3), Bear(4)] 

Bear(2) moves left
Ecosystem status:

[Bear(0), Bear(1), Bear(2), Bear(3), Bear(4)] 



### Ej. 4:

La clase `DynamicArray` descripta a continuación, no soporta el uso de indices negativos usando `__getitem__`. 
Arreglar el método, para que soprte el indexado de Python (faltan los indices negativos) 

In [11]:
class DynamicArray:
    """A dynamic array class akin to a simplified Python list"""
    
    def __init__(self):
        """Create an empty array."""
        self._n = 0
        self._capacity = 1
        self._A = self._make_array(self._capacity)
        
    def __len__(self):
        """Return number of elements stored in the array"""
        return self._n
    
    def __getitem__(self, k):
        if not 0<= k < self._n:
            raise IndexError( 'invalid index' )
        return self._A[k]
        
    def append(self, obj):
        """Add object to end of the array."""
        if self._n == self._capacity:
            self._resize(2 * self._capacity)
        self._A[self._n] = obj
        self._n += 1
    
    def _resize(self, c):
        """Resize internal array to capacity c."""
        B = self._make_array(c)
        for k in range(self._n):
            B[k] = self._A[k]
        self._A = B
        self._capacity = c
        
    def _make_array(self, c):
        """Return new array with capacity c."""
        return (c * ctypes.py_object)()

In [10]:
darray = DynamicArray()
darray.append(4)
darray.append(1)
darray.append(3)
print(darray[-3])
print(darray[-2])
print(darray[-1])
print(darray[0])
print(darray[1])
print(darray[2])

IndexError: invalid index

#### Solución

In [ ]:
    def __getitem__(self, k):
        if 0<= k < self._n:
            return self._A[k]
        else:
            return self._A[k - 1]

In [12]:
darray = DynamicArray()
darray.append(4)
darray.append(1)
darray.append(3)
print(darray[-3])
print(darray[-2])
print(darray[-1])
print(darray[0])
print(darray[1])
print(darray[2])

4
1
3
4
1
3
